# Dendritron Tissue Benchmark v0.4

Single-cell runnable benchmark. Execute the code cell in Google Colab or Jupyter.

In [ ]:
"""
Dendritron Tissue Benchmark v0.4

Shared abstraction without shared mutability.

This benchmark holds the computational primitive constant and compares three
ownership policies over the same learned Boolean Dendritron functions:

1. Shared mutable: one abstraction is reused by all tasks and mutated globally.
2. Independent copies: every task owns a full private abstraction copy.
3. Versioned copy-on-write: tasks share an immutable abstraction until a task
   requires specialization, at which point a content-addressed fork is created
   and only that task is rebound.

The benchmark also tests version deduplication, exact dependency accounting,
localized corruption, integrity detection, and rollback repair.

This is a synthetic architectural benchmark, not a claim of real-world model
superiority.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple
import hashlib
import json
import math

import numpy as np
import pandas as pd

SEED = 19
try:
    OUTPUT_DIR = Path(__file__).resolve().parent
except NameError:
    OUTPUT_DIR = Path.cwd()
RNG = np.random.default_rng(SEED)


def all_binary_inputs(bits: int) -> np.ndarray:
    values = np.arange(2**bits, dtype=np.uint64)
    shifts = np.arange(bits - 1, -1, -1, dtype=np.uint64)
    return ((values[:, None] >> shifts[None, :]) & 1).astype(np.uint8)


def bits_to_index(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.uint8)
    if x.ndim != 2:
        raise ValueError("x must have shape [examples, bits]")
    powers = (2 ** np.arange(x.shape[1] - 1, -1, -1)).astype(np.int64)
    return x.astype(np.int64) @ powers


def balanced_truth_table(bits: int, rng: np.random.Generator) -> np.ndarray:
    n = 2**bits
    table = np.zeros(n, dtype=np.uint8)
    table[rng.choice(n, size=n // 2, replace=False)] = 1
    return table


def specialized_variant(
    base: np.ndarray,
    rng: np.random.Generator,
    swaps: int,
) -> np.ndarray:
    """Create a balanced variant by exchanging `swaps` positive/negative states."""
    base = np.asarray(base, dtype=np.uint8)
    ones = np.flatnonzero(base == 1)
    zeros = np.flatnonzero(base == 0)
    if swaps > min(len(ones), len(zeros)):
        raise ValueError("Too many swaps")
    result = base.copy()
    result[rng.choice(ones, size=swaps, replace=False)] = 0
    result[rng.choice(zeros, size=swaps, replace=False)] = 1
    return result


def table_hash(table: np.ndarray, input_bits: int) -> str:
    payload = bytes([input_bits]) + np.asarray(table, dtype=np.uint8).tobytes()
    return hashlib.sha256(payload).hexdigest()


@dataclass
class BooleanDendritron:
    """An exact local Boolean-function Dendritron.

    The truth table is equivalent to a set of positive minterm branches:

        chi_p(x) = product_j x_j^p_j (1-x_j)^(1-p_j)

    and f(x) = sum_{p: f(p)=1} chi_p(x).
    """

    node_id: str
    name: str
    input_bits: int
    table: np.ndarray
    version: int = 1
    parent_id: str | None = None
    lineage_root: str | None = None

    def __post_init__(self) -> None:
        self.table = np.asarray(self.table, dtype=np.uint8).copy()
        expected = 2**self.input_bits
        if self.table.shape != (expected,):
            raise ValueError(f"Expected table of length {expected}")
        if not np.all((self.table == 0) | (self.table == 1)):
            raise ValueError("Truth table must be binary")
        if self.lineage_root is None:
            self.lineage_root = self.node_id

    def forward(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=np.uint8)
        if x.shape[1] != self.input_bits:
            raise ValueError("Input dimensionality mismatch")
        return self.table[bits_to_index(x)]

    @property
    def branch_count(self) -> int:
        return int(np.sum(self.table))

    @property
    def literal_cost(self) -> int:
        return self.branch_count * self.input_bits

    @property
    def signature(self) -> str:
        return table_hash(self.table, self.input_bits)

    def clone_as(
        self,
        node_id: str,
        table: np.ndarray,
        *,
        name: str | None = None,
    ) -> "BooleanDendritron":
        return BooleanDendritron(
            node_id=node_id,
            name=name or f"{self.name}_v{self.version + 1}",
            input_bits=self.input_bits,
            table=np.asarray(table, dtype=np.uint8),
            version=self.version + 1,
            parent_id=self.node_id,
            lineage_root=self.lineage_root,
        )


@dataclass
class TaskRegion:
    task_id: int
    shared_node_id: str
    local_node: BooleanDendritron
    combiner_node: BooleanDendritron

    def forward(
        self,
        x_shared: np.ndarray,
        x_local: np.ndarray,
        shared_node: BooleanDendritron,
    ) -> np.ndarray:
        shared_value = shared_node.forward(x_shared)
        local_value = self.local_node.forward(x_local)
        pair = np.stack([shared_value, local_value], axis=1)
        return self.combiner_node.forward(pair)


class COWRegistry:
    """Content-addressed, versioned Dendritron abstraction registry."""

    def __init__(self, base_node: BooleanDendritron) -> None:
        self.nodes: Dict[str, BooleanDendritron] = {base_node.node_id: base_node}
        self.snapshot_tables: Dict[str, np.ndarray] = {
            base_node.node_id: base_node.table.copy()
        }
        self.content_index: Dict[Tuple[str, str], str] = {
            (base_node.lineage_root or base_node.node_id, base_node.signature): base_node.node_id
        }
        self.task_refs: Dict[int, str] = {}
        self._counter = 1

    def bind_task(self, task_id: int, node_id: str) -> None:
        if node_id not in self.nodes:
            raise KeyError(node_id)
        self.task_refs[int(task_id)] = node_id

    def refcount(self, node_id: str) -> int:
        return sum(1 for ref in self.task_refs.values() if ref == node_id)

    def dependents(self, node_id: str) -> List[int]:
        return sorted(task for task, ref in self.task_refs.items() if ref == node_id)

    def specialize(self, task_id: int, new_table: np.ndarray) -> dict:
        old_id = self.task_refs[int(task_id)]
        old = self.nodes[old_id]
        new_table = np.asarray(new_table, dtype=np.uint8)
        new_sig = table_hash(new_table, old.input_bits)
        key = (old.lineage_root or old.node_id, new_sig)

        if new_sig == old.signature:
            return {
                "task_id": task_id,
                "old_node_id": old_id,
                "new_node_id": old_id,
                "created_new_version": False,
                "reused_existing_version": True,
                "old_refcount_before": self.refcount(old_id),
                "old_signature_unchanged": True,
            }

        old_refcount = self.refcount(old_id)
        old_signature = old.signature

        if key in self.content_index:
            new_id = self.content_index[key]
            created = False
            reused = True
        else:
            self._counter += 1
            new_id = f"shared_v{self._counter}"
            new_node = old.clone_as(
                new_id,
                new_table,
                name=f"shared_abstraction_v{self._counter}",
            )
            self.nodes[new_id] = new_node
            self.snapshot_tables[new_id] = new_node.table.copy()
            self.content_index[key] = new_id
            created = True
            reused = False

        self.task_refs[int(task_id)] = new_id
        return {
            "task_id": task_id,
            "old_node_id": old_id,
            "new_node_id": new_id,
            "created_new_version": created,
            "reused_existing_version": reused,
            "old_refcount_before": old_refcount,
            "old_refcount_after": self.refcount(old_id),
            "new_refcount_after": self.refcount(new_id),
            "old_signature_unchanged": self.nodes[old_id].signature == old_signature,
        }

    def integrity_failures(self) -> List[str]:
        failures = []
        for node_id, node in self.nodes.items():
            expected = table_hash(self.snapshot_tables[node_id], node.input_bits)
            if node.signature != expected:
                failures.append(node_id)
        return sorted(failures)

    def repair(self, node_id: str) -> None:
        self.nodes[node_id].table = self.snapshot_tables[node_id].copy()

    def active_node_ids(self) -> List[str]:
        return sorted(set(self.task_refs.values()))

    def active_branch_count(self) -> int:
        return sum(self.nodes[node_id].branch_count for node_id in self.active_node_ids())

    def active_literal_cost(self) -> int:
        return sum(self.nodes[node_id].literal_cost for node_id in self.active_node_ids())

    def version_rows(self) -> List[dict]:
        rows = []
        for node_id, node in sorted(self.nodes.items()):
            rows.append(
                {
                    "node_id": node_id,
                    "name": node.name,
                    "version": node.version,
                    "parent_id": node.parent_id,
                    "lineage_root": node.lineage_root,
                    "refcount": self.refcount(node_id),
                    "active": self.refcount(node_id) > 0,
                    "branch_count": node.branch_count,
                    "literal_cost": node.literal_cost,
                    "signature": node.signature,
                }
            )
        return rows


def make_task_regions(
    task_count: int,
    local_bits: int,
    shared_node_id: str,
    rng: np.random.Generator,
) -> Tuple[List[TaskRegion], BooleanDendritron]:
    xor = BooleanDendritron(
        node_id="xor_combiner",
        name="xor_combiner",
        input_bits=2,
        table=np.asarray([0, 1, 1, 0], dtype=np.uint8),
    )
    tasks: List[TaskRegion] = []
    for task_id in range(task_count):
        local = BooleanDendritron(
            node_id=f"local_{task_id}",
            name=f"local_function_{task_id}",
            input_bits=local_bits,
            table=balanced_truth_table(local_bits, rng),
        )
        tasks.append(
            TaskRegion(
                task_id=task_id,
                shared_node_id=shared_node_id,
                local_node=local,
                combiner_node=xor,
            )
        )
    return tasks, xor


def exhaustive_task_inputs(shared_bits: int, local_bits: int) -> Tuple[np.ndarray, np.ndarray]:
    shared = all_binary_inputs(shared_bits)
    local = all_binary_inputs(local_bits)
    x_shared = np.repeat(shared, len(local), axis=0)
    x_local = np.tile(local, (len(shared), 1))
    return x_shared, x_local


def expected_task_output(
    shared_table: np.ndarray,
    local_node: BooleanDendritron,
    combiner: BooleanDendritron,
    x_shared: np.ndarray,
    x_local: np.ndarray,
) -> np.ndarray:
    shared_values = np.asarray(shared_table, dtype=np.uint8)[bits_to_index(x_shared)]
    local_values = local_node.forward(x_local)
    return combiner.forward(np.stack([shared_values, local_values], axis=1))


def evaluate_tasks(
    tasks: Sequence[TaskRegion],
    shared_nodes: Mapping[int, BooleanDendritron],
    target_tables: Mapping[int, np.ndarray],
    x_shared: np.ndarray,
    x_local: np.ndarray,
) -> pd.DataFrame:
    rows = []
    for task in tasks:
        node = shared_nodes[task.task_id]
        pred = task.forward(x_shared, x_local, node)
        target = expected_task_output(
            target_tables[task.task_id],
            task.local_node,
            task.combiner_node,
            x_shared,
            x_local,
        )
        rows.append(
            {
                "task_id": task.task_id,
                "accuracy": float(np.mean(pred == target)),
                "shared_node_id": node.node_id,
            }
        )
    return pd.DataFrame(rows)


def common_storage_cost(
    tasks: Sequence[TaskRegion],
    combiner: BooleanDendritron,
) -> Tuple[int, int]:
    # Local functions are private. The identical immutable combiner is shared once.
    branches = sum(task.local_node.branch_count for task in tasks) + combiner.branch_count
    literals = sum(task.local_node.literal_cost for task in tasks) + combiner.literal_cost
    return branches, literals


def architecture_cost_rows(
    tasks: Sequence[TaskRegion],
    combiner: BooleanDendritron,
    base: BooleanDendritron,
    cow: COWRegistry,
) -> List[dict]:
    common_branches, common_literals = common_storage_cost(tasks, combiner)
    task_count = len(tasks)
    return [
        {
            "architecture": "shared_mutable",
            "active_shared_versions": 1,
            "active_branch_count": common_branches + base.branch_count,
            "active_literal_cost": common_literals + base.literal_cost,
        },
        {
            "architecture": "independent_copies",
            "active_shared_versions": task_count,
            "active_branch_count": common_branches + task_count * base.branch_count,
            "active_literal_cost": common_literals + task_count * base.literal_cost,
        },
        {
            "architecture": "versioned_copy_on_write",
            "active_shared_versions": len(cow.active_node_ids()),
            "active_branch_count": common_branches + cow.active_branch_count(),
            "active_literal_cost": common_literals + cow.active_literal_cost(),
        },
    ]


def run() -> dict:
    rng = np.random.default_rng(SEED)
    task_count = 32
    shared_bits = 4
    local_bits = 4

    base_table = balanced_truth_table(shared_bits, rng)
    base = BooleanDendritron(
        node_id="shared_v1",
        name="shared_abstraction_v1",
        input_bits=shared_bits,
        table=base_table,
    )
    tasks, xor = make_task_regions(task_count, local_bits, base.node_id, rng)
    x_shared, x_local = exhaustive_task_inputs(shared_bits, local_bits)

    # Verify the minterm interpretation exhaustively on the local domain.
    local_domain = all_binary_inputs(shared_bits)
    exact_local_accuracy = float(np.mean(base.forward(local_domain) == base_table))

    # Create one balanced specialization that differs on six of sixteen states.
    single_variant = specialized_variant(base_table, rng, swaps=3)
    single_diff = int(np.sum(single_variant != base_table))
    specialized_task = 7

    target_before = {task.task_id: base_table.copy() for task in tasks}
    target_after_one = {task.task_id: base_table.copy() for task in tasks}
    target_after_one[specialized_task] = single_variant.copy()

    # ------------------------------------------------------------------
    # Shared-mutable control: mutate the one parent for everybody.
    shared_mutable_node = BooleanDendritron(
        node_id="shared_mutable",
        name="shared_mutable",
        input_bits=shared_bits,
        table=base_table,
    )
    mutable_before_sig = shared_mutable_node.signature
    shared_mutable_node.table = single_variant.copy()
    mutable_after_sig = shared_mutable_node.signature
    mutable_mapping = {task.task_id: shared_mutable_node for task in tasks}
    mutable_accuracy = evaluate_tasks(
        tasks, mutable_mapping, target_after_one, x_shared, x_local
    )

    # ------------------------------------------------------------------
    # Independent copies: specialize only one private copy.
    independent_nodes = {
        task.task_id: BooleanDendritron(
            node_id=f"independent_shared_{task.task_id}",
            name=f"independent_shared_{task.task_id}",
            input_bits=shared_bits,
            table=base_table,
        )
        for task in tasks
    }
    independent_signatures_before = {
        task_id: node.signature for task_id, node in independent_nodes.items()
    }
    independent_nodes[specialized_task].table = single_variant.copy()
    independent_accuracy = evaluate_tasks(
        tasks, independent_nodes, target_after_one, x_shared, x_local
    )
    independent_unrelated_nodes_changed = sum(
        independent_nodes[task_id].signature != signature
        for task_id, signature in independent_signatures_before.items()
        if task_id != specialized_task
    )

    # ------------------------------------------------------------------
    # Versioned copy-on-write.
    cow = COWRegistry(
        BooleanDendritron(
            node_id="shared_v1",
            name="shared_abstraction_v1",
            input_bits=shared_bits,
            table=base_table,
        )
    )
    for task in tasks:
        cow.bind_task(task.task_id, "shared_v1")
    cow_signatures_before = {node_id: node.signature for node_id, node in cow.nodes.items()}
    one_fork_event = cow.specialize(specialized_task, single_variant)
    cow_mapping = {
        task.task_id: cow.nodes[cow.task_refs[task.task_id]] for task in tasks
    }
    cow_accuracy = evaluate_tasks(tasks, cow_mapping, target_after_one, x_shared, x_local)
    existing_cow_nodes_changed = sum(
        cow.nodes[node_id].signature != signature
        for node_id, signature in cow_signatures_before.items()
    )

    comparison_rows = []
    for name, frame in [
        ("shared_mutable", mutable_accuracy),
        ("independent_copies", independent_accuracy),
        ("versioned_copy_on_write", cow_accuracy),
    ]:
        spec_acc = float(frame.loc[frame.task_id == specialized_task, "accuracy"].iloc[0])
        sibling = frame.loc[frame.task_id != specialized_task, "accuracy"]
        comparison_rows.append(
            {
                "architecture": name,
                "specialized_task_accuracy": spec_acc,
                "mean_sibling_accuracy": float(sibling.mean()),
                "minimum_sibling_accuracy": float(sibling.min()),
                "mean_all_task_accuracy": float(frame.accuracy.mean()),
                "existing_nodes_changed": (
                    1
                    if name == "shared_mutable" and mutable_before_sig != mutable_after_sig
                    else independent_unrelated_nodes_changed
                    if name == "independent_copies"
                    else existing_cow_nodes_changed
                ),
            }
        )
    comparison = pd.DataFrame(comparison_rows)

    # ------------------------------------------------------------------
    # Family specialization sweep with content-addressed deduplication.
    # 16 tasks specialize into four repeated functional variants.
    variant_tables: List[np.ndarray] = []
    variant_signatures = {table_hash(base_table, shared_bits)}
    for swaps in [1, 2, 3, 4]:
        for _ in range(100):
            candidate = specialized_variant(base_table, rng, swaps=swaps)
            sig = table_hash(candidate, shared_bits)
            if sig not in variant_signatures:
                variant_tables.append(candidate)
                variant_signatures.add(sig)
                break
        else:
            raise RuntimeError("Could not generate distinct specialization variant")

    family_cow = COWRegistry(
        BooleanDendritron(
            node_id="shared_v1",
            name="shared_abstraction_v1",
            input_bits=shared_bits,
            table=base_table,
        )
    )
    for task in tasks:
        family_cow.bind_task(task.task_id, "shared_v1")

    target_family = {task.task_id: base_table.copy() for task in tasks}
    sweep_rows = []
    assignment_tasks = list(range(16))
    event_rows = []
    common_branches, common_literals = common_storage_cost(tasks, xor)

    def append_sweep(step: int) -> None:
        mapping = {
            task.task_id: family_cow.nodes[family_cow.task_refs[task.task_id]]
            for task in tasks
        }
        acc = evaluate_tasks(tasks, mapping, target_family, x_shared, x_local)
        active_versions = len(family_cow.active_node_ids())
        sweep_rows.append(
            {
                "specialized_tasks": step,
                "unique_active_shared_versions": active_versions,
                "mean_accuracy": float(acc.accuracy.mean()),
                "minimum_accuracy": float(acc.accuracy.min()),
                "cow_active_branch_count": common_branches + family_cow.active_branch_count(),
                "cow_active_literal_cost": common_literals + family_cow.active_literal_cost(),
                "shared_mutable_literal_cost": common_literals + base.literal_cost,
                "independent_literal_cost": common_literals + task_count * base.literal_cost,
            }
        )

    append_sweep(0)
    for position, task_id in enumerate(assignment_tasks, start=1):
        variant_index = (task_id // 4) % len(variant_tables)
        variant = variant_tables[variant_index]
        target_family[task_id] = variant.copy()
        event = family_cow.specialize(task_id, variant)
        event["variant_index"] = variant_index
        event_rows.append(event)
        append_sweep(position)

    family_mapping = {
        task.task_id: family_cow.nodes[family_cow.task_refs[task.task_id]]
        for task in tasks
    }
    family_accuracy = evaluate_tasks(tasks, family_mapping, target_family, x_shared, x_local)
    unique_variants_requested = len(
        {table_hash(target_family[task_id], shared_bits) for task_id in assignment_tasks}
    )
    newly_created_versions = sum(bool(row["created_new_version"]) for row in event_rows)
    reused_events = sum(bool(row["reused_existing_version"]) for row in event_rows)

    cost_rows = architecture_cost_rows(tasks, xor, base, family_cow)
    costs = pd.DataFrame(cost_rows)

    # ------------------------------------------------------------------
    # Localized structural corruption and repair.
    base_node_id = "shared_v1"
    base_dependents = family_cow.dependents(base_node_id)
    base_nondependents = sorted(set(range(task_count)) - set(base_dependents))
    pre_damage_signatures = {
        node_id: node.signature for node_id, node in family_cow.nodes.items()
    }
    pre_damage_accuracy = evaluate_tasks(
        tasks, family_mapping, target_family, x_shared, x_local
    ).set_index("task_id")

    # Simulate one branch/state corruption in the shared base abstraction.
    corruption_index = int(np.flatnonzero(family_cow.nodes[base_node_id].table == 0)[0])
    family_cow.nodes[base_node_id].table[corruption_index] = 1
    failures = family_cow.integrity_failures()
    damaged_mapping = {
        task.task_id: family_cow.nodes[family_cow.task_refs[task.task_id]]
        for task in tasks
    }
    damaged_accuracy = evaluate_tasks(
        tasks, damaged_mapping, target_family, x_shared, x_local
    ).set_index("task_id")

    dependent_accuracy_after_damage = float(
        damaged_accuracy.loc[base_dependents, "accuracy"].mean()
    )
    nondependent_accuracy_after_damage = float(
        damaged_accuracy.loc[base_nondependents, "accuracy"].mean()
    )
    nondependent_predictions_changed = int(
        np.sum(
            ~np.isclose(
                damaged_accuracy.loc[base_nondependents, "accuracy"].to_numpy(),
                pre_damage_accuracy.loc[base_nondependents, "accuracy"].to_numpy(),
            )
        )
    )

    family_cow.repair(base_node_id)
    repaired_failures = family_cow.integrity_failures()
    repaired_mapping = {
        task.task_id: family_cow.nodes[family_cow.task_refs[task.task_id]]
        for task in tasks
    }
    repaired_accuracy = evaluate_tasks(
        tasks, repaired_mapping, target_family, x_shared, x_local
    ).set_index("task_id")
    other_nodes_changed_during_repair = sum(
        family_cow.nodes[node_id].signature != signature
        for node_id, signature in pre_damage_signatures.items()
        if node_id != base_node_id
    )

    repair_rows = [
        {
            "damaged_node_id": base_node_id,
            "dependent_task_count": len(base_dependents),
            "nondependent_task_count": len(base_nondependents),
            "integrity_failures_detected": len(failures),
            "failed_node_identified_exactly": failures == [base_node_id],
            "mean_dependent_accuracy_after_damage": dependent_accuracy_after_damage,
            "mean_nondependent_accuracy_after_damage": nondependent_accuracy_after_damage,
            "nondependent_tasks_with_accuracy_change": nondependent_predictions_changed,
            "mean_accuracy_after_repair": float(repaired_accuracy.accuracy.mean()),
            "integrity_failures_after_repair": len(repaired_failures),
            "other_nodes_changed_during_repair": other_nodes_changed_during_repair,
        }
    ]
    repair_frame = pd.DataFrame(repair_rows)

    # Hard benchmark assertions.
    assert exact_local_accuracy == 1.0
    assert float(mutable_accuracy.loc[mutable_accuracy.task_id == specialized_task, "accuracy"].iloc[0]) == 1.0
    assert float(mutable_accuracy.loc[mutable_accuracy.task_id != specialized_task, "accuracy"].mean()) == 0.625
    assert float(independent_accuracy.accuracy.min()) == 1.0
    assert float(cow_accuracy.accuracy.min()) == 1.0
    assert existing_cow_nodes_changed == 0
    assert bool(one_fork_event["old_signature_unchanged"])
    assert float(family_accuracy.accuracy.min()) == 1.0
    assert len(family_cow.active_node_ids()) == 5
    assert newly_created_versions == 4
    assert reused_events == 12
    assert failures == [base_node_id]
    assert nondependent_predictions_changed == 0
    assert float(repaired_accuracy.accuracy.min()) == 1.0
    assert other_nodes_changed_during_repair == 0

    # Save outputs.
    comparison.to_csv(OUTPUT_DIR / "dendritron_v0_4_architecture_comparison.csv", index=False)
    mutable_accuracy.assign(architecture="shared_mutable").to_csv(
        OUTPUT_DIR / "dendritron_v0_4_shared_mutable_tasks.csv", index=False
    )
    independent_accuracy.assign(architecture="independent_copies").to_csv(
        OUTPUT_DIR / "dendritron_v0_4_independent_tasks.csv", index=False
    )
    cow_accuracy.assign(architecture="versioned_copy_on_write").to_csv(
        OUTPUT_DIR / "dendritron_v0_4_cow_tasks.csv", index=False
    )
    pd.DataFrame(sweep_rows).to_csv(
        OUTPUT_DIR / "dendritron_v0_4_specialization_sweep.csv", index=False
    )
    pd.DataFrame(event_rows).to_csv(
        OUTPUT_DIR / "dendritron_v0_4_fork_events.csv", index=False
    )
    costs.to_csv(OUTPUT_DIR / "dendritron_v0_4_storage_comparison.csv", index=False)
    pd.DataFrame(family_cow.version_rows()).to_csv(
        OUTPUT_DIR / "dendritron_v0_4_version_graph.csv", index=False
    )
    family_accuracy.to_csv(
        OUTPUT_DIR / "dendritron_v0_4_family_accuracy.csv", index=False
    )
    repair_frame.to_csv(OUTPUT_DIR / "dendritron_v0_4_repair.csv", index=False)

    summary = {
        "seed": SEED,
        "task_count": task_count,
        "shared_bits": shared_bits,
        "local_bits": local_bits,
        "local_dendritron_exact_accuracy": exact_local_accuracy,
        "single_specialization": {
            "specialized_task": specialized_task,
            "shared_states_changed": single_diff,
            "shared_mutable_specialized_accuracy": float(
                mutable_accuracy.loc[
                    mutable_accuracy.task_id == specialized_task, "accuracy"
                ].iloc[0]
            ),
            "shared_mutable_mean_sibling_accuracy": float(
                mutable_accuracy.loc[
                    mutable_accuracy.task_id != specialized_task, "accuracy"
                ].mean()
            ),
            "independent_mean_accuracy": float(independent_accuracy.accuracy.mean()),
            "cow_mean_accuracy": float(cow_accuracy.accuracy.mean()),
            "cow_existing_nodes_changed": existing_cow_nodes_changed,
            "cow_original_parent_signature_unchanged": bool(
                one_fork_event["old_signature_unchanged"]
            ),
        },
        "family_specialization": {
            "specialized_tasks": len(assignment_tasks),
            "unique_variants_requested": unique_variants_requested,
            "new_versions_created": newly_created_versions,
            "existing_versions_reused": reused_events,
            "active_shared_versions": len(family_cow.active_node_ids()),
            "mean_accuracy": float(family_accuracy.accuracy.mean()),
            "minimum_accuracy": float(family_accuracy.accuracy.min()),
        },
        "repair": repair_rows[0],
        "storage": costs.to_dict(orient="records"),
    }
    with open(OUTPUT_DIR / "dendritron_v0_4_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("DENDRITRON TISSUE BENCHMARK v0.4")
    print("=" * 72)
    print(f"Exact local Dendritron accuracy: {exact_local_accuracy:.2%}")
    print(f"Single specialization changed {single_diff}/16 shared states")
    print("\nOwnership-policy comparison")
    print(comparison.to_string(index=False))
    print("\nFamily specialization")
    print(
        f"16 task requests collapsed into {len(family_cow.active_node_ids())} active "
        f"shared versions (base plus {newly_created_versions} unique forks)."
    )
    print(
        f"Final family accuracy: mean={family_accuracy.accuracy.mean():.2%}, "
        f"min={family_accuracy.accuracy.min():.2%}"
    )
    print("\nActive storage after family specialization")
    print(costs.to_string(index=False))
    print("\nLocalized damage and repair")
    print(repair_frame.to_string(index=False))
    print("\nSummary written to dendritron_v0_4_summary.json")
    return summary


if __name__ == "__main__":
    run()
